In [1]:
import pandas as pd

df = pd.read_csv("processed_clause_dataset_clean.csv")

In [2]:
contracts = (
    df.groupby("contract_title")["context"]
    .first()
    .reset_index()
)

In [3]:
# Creatinbg Text Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [5]:
documents = []
for _, row in contracts.iterrows():

    chunks = splitter.split_text(
        row["context"]
    )

    for chunk in chunks:

        documents.append({
            "contract_title": row["contract_title"],
            "chunk_text": chunk
        })

In [6]:
len(documents)

39020

In [7]:
# Generate Embeddings
from sentence_transformers import SentenceTransformer

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
texts = [
    doc["chunk_text"]
    for doc in documents
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/1220 [00:00<?, ?it/s]

In [11]:
print(embeddings.shape)

(39020, 384)


In [12]:
# Create FAISS Index
import faiss
import numpy as np

embeddings = np.array(
    embeddings
).astype("float32")

In [13]:
index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

In [14]:
print(index.ntotal)

39020


In [15]:
# Retrieval Function
def retrieve_chunks(
    query,
    top_k=5
):

    query_embedding = (
        embedding_model
        .encode([query])
        .astype("float32")
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            documents[idx]["chunk_text"]
        )

    return results

In [16]:
retrieve_chunks(
    "termination clause"
)

['32.4. the following Clauses shall survive termination for whatever cause             of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34             inclusive.\n\n33.   Rights Upon Termination\n\n      Upon termination of this Agreement and for a period of six (6) months       thereafter, the Publishers will have the following rights and obligations:',
 '8.       Termination.          (a) Upon the  occurrence  of a  material  breach or  default  as to any          obligation,  term or provision contained herein by either party and the          failure of the breaching  party to promptly  pursue (within thirty (30)          days after  receiving  written  notice  thereof from the  non-breaching          party) a reasonable remedy designed to cure (in the reasonable judgment          of the  non-breaching  party) such  material  breach or  default,  this          Agreement  may be  terminated  by the  non-breaching  party  by  giving          written notice of termination

In [35]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv(override=True)

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "What is a governing law clause?"
        }
    ]
)

print(response.choices[0].message.content)

A governing law clause, also known as a choice of law clause or jurisdiction clause, is a provision in a contract that specifies which laws will apply to the agreement and which courts will have jurisdiction over any disputes that may arise. This clause is essential in contracts involving parties from different countries or jurisdictions, as it helps to clarify which laws will govern the contract and where any disputes will be resolved.

A typical governing law clause might state something like:

"This Agreement shall be governed by and construed in accordance with the laws of [State/Country], and any disputes arising out of or related to this Agreement shall be resolved through [dispute resolution process, e.g., arbitration or litigation] in [State/Country]."

The governing law clause serves several purposes:

1. ** Certainty**: It provides clarity on which laws will apply to the contract, reducing uncertainty and potential conflicts.
2. **Predictability**: It allows parties to antici

In [37]:
print("llm" in globals())

True


In [38]:
# RAG Function
def ask_contract(query):

    retrieved_chunks = retrieve_chunks(
        query,
        top_k=5
    )

    context = "\n\n".join(
        retrieved_chunks
    )

    prompt = f"""
You are a legal AI assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [39]:
print("retrieve_chunks" in globals())
print("index" in globals())
print("documents" in globals())
print("llm" in globals())

True
True
True
True


In [42]:
answer = ask_contract(
    "What is the termination clause?"
 )

print(answer)

The termination clause is found in two sections: 

Section 8: Termination, which states that upon the occurrence of a material breach or default by either party and the failure to remedy such breach within 30 days, the non-breaching party may terminate the agreement by giving written notice, effective immediately.

Additionally, Section 33: Rights Upon Termination, and Clause 32.4, which lists the clauses that shall survive termination, and Section 8(g) also describe the effects and rights after termination.
